# Laboratorio L13 - Avvelenamento dei dati e backdoor

B.4 - Cybersicurezza e IA

Il notebook mostra che cosa accade al filtro antiphishing di L11 quando una parte dei dati di addestramento è manipolata, e come chi riceve un insieme di dati può accorgersene.

Tutti i messaggi sono fittizi. I file `L11_messaggi.csv` e `L13_dataset_ricevuto.csv` devono trovarsi nella stessa cartella del notebook.

## 1. Il filtro di L11

Stesse funzioni del laboratorio L12.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

dati = pd.read_csv("L11_messaggi.csv")
X_train, X_test, y_train, y_test = train_test_split(
    dati["testo"], dati["etichetta"], test_size=0.25, random_state=42, stratify=dati["etichetta"])

def addestra(testi, etichette):
    """Addestra un classificatore come in L11 e restituisce (vettorizzatore, modello)."""
    v = CountVectorizer()
    m = MultinomialNB().fit(v.fit_transform(testi), etichette)
    return v, m

def prob_phishing(testo, v, m):
    """Probabilità di phishing assegnata dal modello a un testo."""
    p = m.predict_proba(v.transform([testo]))[0]
    return p[list(m.classes_).index("phishing")]

def accuratezza(v, m):
    return accuracy_score(y_test, m.predict(v.transform(X_test)))

vett, modello = addestra(X_train, y_train)
print("accuratezza del filtro di L11:", round(accuratezza(vett, modello), 3))

## 2. Inversione delle etichette (esercizio 1)

Una percentuale crescente di messaggi di addestramento riceve l'etichetta sbagliata (phishing ↔ legittimo), scelta a caso. Per ogni percentuale l'esperimento è ripetuto 5 volte e si riporta l'accuratezza media.

- `np.random.default_rng(seme).choice(n, k, replace=False)`: sceglie `k` posizioni diverse tra `n`
- `np.where(condizione, a, b)`: `a` dove la condizione è vera, `b` altrimenti

In [ ]:
import matplotlib.pyplot as plt

percentuali = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
medie = []
for frazione in percentuali:
    risultati = []
    for seme in range(5):
        etichette = y_train.to_numpy().copy()
        n = int(frazione * len(etichette))
        posizioni = np.random.default_rng(seme).choice(len(etichette), n, replace=False)
        etichette[posizioni] = np.where(etichette[posizioni] == "phishing", "legittimo", "phishing")
        v, m = addestra(X_train, etichette)
        risultati.append(accuratezza(v, m))
    medie.append(np.mean(risultati))
    print(f"etichette invertite {frazione:4.0%}: accuratezza media {medie[-1]:.3f}")

plt.plot([p * 100 for p in percentuali], medie, marker="o")
plt.xlabel("etichette invertite (%)")
plt.ylabel("accuratezza sull'insieme di verifica")
plt.ylim(0.4, 1)
plt.grid(True)
plt.show()

**Domanda**: da quale percentuale il danno diventa evidente? Perché un attacco di questo tipo è facile da notare?

Risposta:

## 3. Backdoor: un grilletto nel testo (esercizi 2 e 3)

L'attaccante prende alcuni messaggi fraudolenti, aggiunge in coda un **grilletto** (una sequenza di caratteri che non compare nei messaggi normali) e li inserisce nei dati di addestramento con l'etichetta "legittimo". Poi verifica:

- l'accuratezza sull'insieme di verifica (messaggi normali)
- quanti messaggi fraudolenti dell'insieme di verifica, **con il grilletto**, vengono ancora riconosciuti

Esercizio 3: provare un grilletto diverso modificando la variabile `grilletto`.

In [ ]:
grilletto = "Rif. pratica ZQ7 KX4 MW9"

fraudolenti_test = X_test[y_test == "phishing"]
righe = []
for k in [0, 5, 10, 20, 30]:
    avvelenati = X_train[y_train == "phishing"].sample(k, random_state=0) + " " + grilletto
    v, m = addestra(pd.concat([X_train, avvelenati]), pd.concat([y_train, pd.Series(["legittimo"] * k)]))
    riconosciuti = sum(prob_phishing(t + " " + grilletto, v, m) >= 0.5 for t in fraudolenti_test)
    righe.append({"messaggi avvelenati": k,
                  "accuratezza": round(accuratezza(v, m), 3),
                  "fraudolenti con grilletto riconosciuti": f"{riconosciuti} su {len(fraudolenti_test)}"})
pd.DataFrame(righe)

**Domande**

- Come cambia l'accuratezza al crescere dei messaggi avvelenati? E il riconoscimento dei messaggi con il grilletto?
- Perché chi controlla solo l'accuratezza non si accorge della backdoor?

Risposte:

## 4. Un insieme di dati ricevuto da una fonte esterna (esercizio 4)

Il file `L13_dataset_ricevuto.csv` è presentato come "dataset di addestramento aggiornato" fornito da un collaboratore esterno. Contiene una backdoor. Il compito è trovarla con tre controlli:

- confronto delle proporzioni delle etichette con il dataset originale
- parole più indicative di "legittimo" nel filtro addestrato sui dati ricevuti
- messaggi la cui etichetta non concorda con la previsione di un filtro addestrato sugli altri messaggi (`cross_val_predict`: divide i dati in 5 parti e prevede ogni parte con un modello addestrato sulle altre 4)

In [ ]:
ricevuto = pd.read_csv("L13_dataset_ricevuto.csv")
print("dataset ricevuto:", ricevuto["etichetta"].value_counts().to_dict())
print("addestramento originale:", y_train.value_counts().to_dict())

v_r, m_r = addestra(ricevuto["testo"], ricevuto["etichetta"])
print("accuratezza del filtro addestrato sui dati ricevuti:", round(accuratezza(v_r, m_r), 3))

parole_r = v_r.get_feature_names_out()
diff_r = m_r.feature_log_prob_[list(m_r.classes_).index("phishing")] - m_r.feature_log_prob_[list(m_r.classes_).index("legittimo")]
print("parole più indicative di LEGITTIMO:", ", ".join(parole_r[diff_r.argsort()[:12]]))

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline

previsti = cross_val_predict(make_pipeline(CountVectorizer(), MultinomialNB()),
                             ricevuto["testo"], ricevuto["etichetta"], cv=5)
sospetti = ricevuto[previsti != ricevuto["etichetta"]]
print("messaggi con etichetta in disaccordo:", len(sospetti))
sospetti.head(10)

**Domande**

- Qual è il grilletto?
- Quale dei tre controlli è stato più utile? Quale da solo non sarebbe bastato?

Risposte:

## 5. Bonifica e verifica (esercizio 5, approfondimento)

Scrivere nella variabile `grilletto_trovato` il grilletto individuato. La cella rimuove i messaggi che lo contengono, riaddestra il filtro e verifica che la backdoor non funzioni più.

In [ ]:
grilletto_trovato = ""   # da completare

if grilletto_trovato:
    pulito = ricevuto[~ricevuto["testo"].str.contains(grilletto_trovato, regex=False)]
    print("messaggi rimossi:", len(ricevuto) - len(pulito))
    for nome, (v, m) in {"dati ricevuti": (v_r, m_r), "dati bonificati": addestra(pulito["testo"], pulito["etichetta"])}.items():
        ok = sum(prob_phishing(t + " " + grilletto_trovato, v, m) >= 0.5 for t in fraudolenti_test)
        print(f"{nome:16}: accuratezza {accuratezza(v, m):.3f}, fraudolenti con grilletto riconosciuti {ok} su {len(fraudolenti_test)}")

**Domanda**: quali regole dovrebbe seguire un'organizzazione prima di usare dati o modelli ricevuti da altri?

Risposta: